<a href="https://colab.research.google.com/github/TPVinnie/aws-converse-api/blob/master/Amazon%20Nova/amazon-converse-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting Started with AWS Converse API


In [ ]:
!pip install -qq boto3==1.41.1
print("boto3 version:", boto3.__version__)

In [ ]:
import boto3
import json
from botocore.exceptions import ClientError
from botocore.config import Config

# Optional: extend timeout for image processing (recommended for Nova)
config = Config(connect_timeout=60, read_timeout=3600)

session = boto3.Session()
bedrock = session.client(
    service_name="bedrock-runtime",
    region_name="us-east-1",
    config=config
)

### CODE 1 - A basic call to the Converse API

In [ ]:
message_list = []

# Your initial user message
initial_message = {
    "role": "user",
    "content": [
        { "text": "How are you today?" }
    ],
}

message_list.append(initial_message)

try:
    response = bedrock.converse(
        modelId="amazon.nova-lite-v1:0",
        messages=message_list,
        inferenceConfig={
            "maxTokens": 2000,
            "temperature": 0.0,
            "topP": 0.9
        }
    )
    response_message = response["output"]["message"]
    print(json.dumps(response_message, indent=4))

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke model. Reason: {e}")
    exit(1)


###  CODE 2 - Alternating user and assistant messages

####  Alternate between messages from the “user” and “assistant” roles.

In [ ]:
message_list.append(response_message)
print(json.dumps(message_list, indent=4))

### CODE 3 - Including an image in a message

![Alt Text](https://github.com/TPVinnie/aws-converse-api/blob/master/Amazon%20Nova/image.webp?raw=1)

In [ ]:
# 1️⃣ Initial user text message
message_list.append({
    "role": "user",
    "content": [{"text": "How are you today?"}]
})

# 2️⃣ Image input followed by a prompt
with open("image.webp", "rb") as image_file:
    image_bytes = image_file.read()

image_message = {
    "role": "user",
    "content": [
        {"text": "Image 1:"},
        {
            "image": {
                "format": "webp",
                "source": {"bytes": image_bytes}
            }
        },
        {"text": "Please describe the image."}
    ]
}

message_list.append(image_message)

model_id = "amazon.nova-lite-v1:0"  # or nova‑pro for higher capability

try:
    response = bedrock.converse(
        modelId=model_id,
        messages=message_list,
        inferenceConfig={
            "maxTokens": 2000,
            "temperature": 0.0,
            "topP": 0.9
        }
    )
    response_message = response["output"]["message"]
    print(json.dumps(response_message, indent=4))

    # Continue the conversation
    message_list.append(response_message)

except ClientError as e:
    print("AWS API error:", e.response.get("Error", {}).get("Message"))
except Exception as e:
    print("Unexpected error:", e)


### CODE 4 - Setting a system prompt


In [ ]:
# (Assuming you already have previous messages appended here)

# Add the summary request message
summary_message = {
    "role": "user",
    "content": [
        {"text": "Can you please summarize our conversation so far?"}
    ],
}
message_list.append(summary_message)

system_prompts = [
    {"text": "You must respond to all requests in poaragraph."}
]

inference_config = {
    "maxTokens": 2000,
    "temperature": 0.0,
    "topP": 0.9
}

try:
    response = bedrock.converse(
        modelId="amazon.nova-lite-v1:0",  # or nova‑pro, nova‑premier as needed
        messages=message_list,
        system=system_prompts,
        inferenceConfig=inference_config
        # no additionalModelRequestFields unless you need model-specific params like topK
    )
    response_message = response["output"]["message"]
    print(json.dumps(response_message, indent=4))

    # Save assistant response into conversation history
    message_list.append(response_message)
    #print(message_list)

    print("Stop Reason:", response.get("stopReason"))
    print("Usage:", json.dumps(response.get("usage", {}), indent=4))

except ClientError as e:
    print("AWS API error:", e.response.get("Error", {}).get("Message"))
except Exception as e:
    print("Unexpected error:", e)


## Getting started with converse stream

In [ ]:
message_list = []

# Your initial user message
initial_message = {
    "role": "user",
    "content": [
        { "text": "Can you write a detailed story about AI in space" }
    ],
}

message_list.append(initial_message)

try:
    response = bedrock.converse_stream(
        modelId="amazon.nova-lite-v1:0",
        messages=message_list,
        inferenceConfig={
            "maxTokens": 2000,
            "temperature": 0.0,
            "topP": 0.9
        }
    )

    # Stream the response chunks as they arrive
    stream = response.get("stream")
    if stream:
        for event in stream:
            if "contentBlockDelta" in event:
                text_chunk = event["contentBlockDelta"]["delta"]["text"]
                print(text_chunk, end="", flush=True)
            elif "messageStop" in event:
                print()  # newline at end
                print(f"\nStop reason: {event['messageStop']['stopReason']}")

except (ClientError, Exception) as e:
    print(f"ERROR: Can't invoke model. Reason: {e}")
    exit(1)